In [ ]:
import os
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from astra.src.embedding import AstraEmbedding
from astra.src.preprocessing import *

In [17]:
def test_data_loader():
    # 1. Configuration for the test
    # REPLACE THIS with the actual path to your TFRecords folder
    SOURCE_DIR = "/home/nvidia/workplace/dataset/test/val/" 
    
    BATCH_SIZE = 4  # Keep it small for a quick printout test
    NUM_GLOBAL_VIEWS = 3
    NUM_LOCAL_VIEWS = 6
    
    # Example maximum lengths (adjust to match your actual logic)
    gv_maxlens={'g': 180, 'r': 200, 'i': 20}
    lv_maxlens_list=[
    {'g': 45, 'r': 50, 'i': 5},  
    {'g': 50, 'r': 55, 'i': 2},   
    {'g': 30, 'r': 50, 'i': 10},  
    {'g': 40, 'r': 40, 'i': 8}    ]
    build_seq_len=400
    apply_noise_list= [True, True, False, False, True]
    noise_levels_list= [0.01,   0.01,  0.05,  0.05,   0.02]
    apply_binning_list= [True, False, True, True, False]
    apply_outlier_list= [True, True, False, False, True]
    bin_widths_list= [5,     5,    5,    5,     5]
    drop_rates_list= [0.05,   0.01,  0.01,  0.01,   0.05]        

    print("Initializing Data Loader...")
    try:
        dataset = clustering_data_loader(
                                                    source=SOURCE_DIR, 
                                                    batch_size=BATCH_SIZE, 
                                                    buffer_size=100,
                                                    seed=1024,
                                                    gv_maxlens=gv_maxlens, 
                                                    lv_maxlens_list=lv_maxlens_list, 
                                                    num_local_views=NUM_LOCAL_VIEWS,
                                                    num_global_views=NUM_GLOBAL_VIEWS,
                                                    apply_noise_list=apply_noise_list, 
                                                    noise_levels_list=noise_levels_list, 
                                                    apply_binning_list=apply_binning_list, 
                                                    apply_outlier_list=apply_outlier_list,
                                                    bin_widths_list=bin_widths_list, 
                                                    drop_rates_list=drop_rates_list, 
                                                    build_seq_len=build_seq_len 
                                                )
    except ValueError as e:
        print(e)
        return

    print("\nFetching one batch using .take(1) ...\n")
    print(f"\n'g': {np.log10(4746.48)}, 'r': {np.log10(6366.38)}, 'i': {np.log10(7829.03)}")
    # 2. Extract exactly one batch
    for batch in dataset.take(1):
        
        print(f"Total views in tuple: {len(batch)} (Expected: 9)")
        
        # 3. Iterate through the 9 views (3 GVs + 6 LVs)
        for i, view_dict in enumerate(batch):
            
            # Determine if this is a Global or Local view
            view_type = "GLOBAL VIEW" if i < NUM_GLOBAL_VIEWS else "LOCAL VIEW"
            
            print(f"\n{'='*50}")
            print(f" View {i + 1} : {view_type}")
            print(f"{'='*50}")
            
            # Print the shapes and dtypes of the tensors
            for key, tensor in view_dict.items():
                print(f"  {key:10s} -> Shape: {tensor.shape}, Dtype: {tensor.dtype.name}")
            
            # 4. Inspect the ACTUAL values of the FIRST sample in this batch (Index 0)
            print("\n  [ Data Inspection for Batch Sample 0 ]")
            
            # Extract numpy arrays for easy viewing
            mask = view_dict['mask'][0].numpy()
            mag = view_dict['input'][0].numpy().flatten()
            times = view_dict['times'][0].numpy().flatten()
            band = view_dict['band_info'][0].numpy().flatten()
            print("\nTotal Unique Bands:", np.unique(band, return_counts=True))
            
            seq_len = len(mask)
            # Total masked elements (Augmentation Drops + Padding)
            total_masked = np.sum(mask == 1.0)
            
            # --- NEW PADDING DETECTION LOGIC ---
            # Find all indices where the mask is 0.0 (valid, untouched data)
            valid_indices = np.where(mask == 0.0)[0]
            
            if len(valid_indices) > 0:
                # The boundary of the real sequence is the last valid index
                last_valid_idx = valid_indices[-1]
                
                # Everything AFTER this index is the contiguous padding block added by padded_batch
                num_padded = seq_len - 1 - last_valid_idx
            else:
                # Fallback if the entire sequence was somehow masked out
                num_padded = 0
            
            # Identify augmented drops: where mask == 1.0 but time != 0.0
            num_augmented_drops = total_masked - num_padded
            
            print(f"  Sequence Length dynamically padded to : {seq_len}")
            print(f"  Total Masked Steps : {total_masked} ({total_masked/seq_len*100:.1f}%)")
            print(f"    -> Steps masked by Augmentation : {num_augmented_drops}")
            print(f"    -> Steps masked by Padding      : {num_padded}")
            
            # Print the beginning of the sequence (should be real data, mask mostly 0.0)
            print(f"\n  First 5 steps (Real Data / Augmented):")
            print(f"    Mag   : {mag[:5]}")
            print(f"    Times : {times[:5]}")
            print(f"    Mask  : {mask[:5]}")
            print(f"    Band  : {band[:5]}")
            
            # Print the end of the sequence (should be purely padded data: mask=1.0, mag=0.0, time=0.0)
            if num_padded > 0:
                print(f"\n  Last 5 steps (Padded Data):")
                print(f"    Mag   : {mag[-5:]}")
                print(f"    Times : {times[-5:]}")
                print(f"    Mask  : {mask[-5:]}")
                print(f"    Band  : {band[-5:]}")
            else:
                print("\n  Last 5 steps (No padding needed for this sample):")
                print(f"    Mag   : {mag[-5:]}")
                print(f"    Times : {times[-5:]}")
                print(f"    Mask  : {mask[-5:]}")
                print(f"    Band  : {band[-5:]}")

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)
            
test_data_loader()

### Data Loader for Contrastive Learning (Anchor, Positive, Negative)

In [ ]:
#
# This script demonstrates how to use the contrastive_data_loader function
#
n_views = 3
batch_size = 200
apply_white_noise=[False, True, True]
apply_binning=[False, False, True]
apply_outlier=[False, False, True]
maxlens=[{'g': 100, 'r': 200, 'i': 50}, {'g': 100, 'r': 150, 'i': 50}, {'g': 100, 'r': 200, 'i': 50}] 
bin_widths = [5, 5, 5]
drop_rates = [0.0, 0.0, 0.60]
noise_levels = [0.10, 0.0, 0.10] 
buffer_size = 10000  
build_seq_len=tf.cast(sum(maxlens[0].values()), tf.int32)  # Final fixed length for sequences 

path_to_read = "/media3/majumder/dataset/resampled_multi-class/val/"
#
loaders = contrastive_data_loader( source=path_to_read,
                                    n_views=n_views, 
                                    seed=np.random.randint(1024), 
                                    batch_size=batch_size,
                                    build_seq_len=build_seq_len,
                                    apply_white_noise=apply_white_noise,
                                    noise_levels=noise_levels,
                                    apply_binning=apply_binning,
                                    apply_outlier=apply_outlier,
                                    maxlens=maxlens,
                                    bin_widths=bin_widths,
                                    drop_rates=drop_rates,
                                    buffer_size=buffer_size
                                )

# im1, im2, im3 = next(iter(loaders))
# print(im1, im2, im3)
# for anchor, positive, negative in loaders:
#   print(anchor.keys(), anchor.values(), positive, negative)
#   break

# pbar_train = tqdm(loaders)
# for batch in pbar_train:
#     tf.print(batch[0]['input'].shape, batch[0]['times'].shape, batch[0]['band_info'].shape, batch[0]['mask'].shape)
#     tf.print(batch[1]['input'].shape, batch[1]['times'].shape, batch[1]['band_info'].shape, batch[1]['mask'].shape)
#     tf.print(batch[2]['input'].shape, batch[2]['times'].shape, batch[2]['band_info'].shape, batch[2]['mask'].shape)
#     tf.print(batch[1]['mask'][1, 297: 303])
#     tf.print(batch[1]['times'][1, 297: 303,:])
#     break

In [ ]:
strategy = tf.distribute.get_strategy()
distributed_dataset = strategy.experimental_distribute_dataset(loaders)
pbar_train = tqdm(distributed_dataset)
for batch in pbar_train:
    tf.print(batch[0]['input'].shape, batch[0]['times'].shape, batch[0]['band_info'].shape, batch[0]['mask'].shape)
    tf.print(batch[1]['input'].shape, batch[1]['times'].shape, batch[1]['band_info'].shape, batch[1]['mask'].shape)
    tf.print(batch[2]['input'].shape, batch[2]['times'].shape, batch[2]['band_info'].shape, batch[2]['mask'].shape)
    tf.print(batch[1]['mask'][1, 297: 303], batch[0]['mask'][1, 297: 303], batch[2]['mask'][1, 297: 303])
    tf.print(batch[1]['times'][1, 297: 303,:] , batch[0]['times'][1, 297: 303,:], batch[2]['times'][1, 297: 303,:])
    break

In [ ]:
for anchor, positive, negative in loaders:
  print(anchor, positive, negative)
  break

### Generate Time-series embeddings for dart

In [ ]:
# Define dimensions
d_model = 512

In [ ]:
emb = AstraEmbedding(d_model=d_model, base=10000, rate=0.1, use_band_info=True, use_drop=True, mjd=True)
tensor = emb(negative['input'], negative['times'], negative['band_info'])
print(tensor.shape)
print(tensor)

In [ ]:
from cdshealpix import healpix_to_skycoord

skycoord= healpix_to_skycoord(1525403389526329698, depth=29)
print(skycoord)